In [1]:

import requests
import pandas as pd
from pathlib import Path
import math

In [2]:

url = "https://previous-runs-api.open-meteo.com/v1/forecast"
# coordinates for Tõravere weather station
lat, lon = 58.2642, 26.4614
timezone = "Europe/Tallinn"

weather_models = {
    "ECMWF": "ecmwf_ifs025",
    "GFS": "ncep_gfs_global",
    "ICON_EU": "icon_eu"
}

weather_variables = {
    "temperature_2m_previous_day1": "temperature",
    "precipitation_previous_day1": "precipitation",
    "wind_speed_10m_previous_day1": "wind_speed"
}

folder = Path.cwd()
raw_folder = folder / "raw"
processed_folder = folder / "processed"

In [3]:

all_data = []

for year in [2024, 2025, 2026]:
    for name, model in weather_models.items():

        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": f"{year}-06-01",
            "end_date": f"{year}-08-31",
            "hourly": ",".join(weather_variables.keys()),
            "models": model,
            "timezone": timezone,
            "wind_speed_unit": "ms"
        }

        response = requests.get(
            url,
            params=params,
            timeout=120
        )
        response.raise_for_status()

        data = response.json()
        df = pd.DataFrame(data["hourly"])

        df["model"] = name
        df["year"] = year

        all_data.append(df)

combined_data = pd.concat(all_data, ignore_index=True)

combined_data.to_csv(
    raw_folder / "forecast_data_raw.csv",
    index=False
)

In [10]:

def read_actual_measurements(folder, fail, veeru_nimi):
    df = pd.read_csv(
        folder / fail,
        sep=";",
        decimal=","
    )

    # dates in Keskkonnaagentuur data are in UTC
    time_utc = pd.to_datetime({
        "year": df.iloc[:, 1],
        "month": df.iloc[:, 2],
        "day": df.iloc[:, 3],
        "hour": df.iloc[:, 4]
    }, utc=True)

    time = time_utc.dt.tz_convert(timezone)

    result = pd.DataFrame({
        "time": time.dt.tz_localize(None),
        veeru_nimi: df.iloc[:, 5]
    })

    # Keep summer months of 2024–2026
    result = result[
        result["time"].dt.year.isin([2024, 2025, 2026]) &
        result["time"].dt.month.isin([6, 7, 8])
    ]

    return result

In [11]:

temp = read_actual_measurements(raw_folder, "measured_temperature.csv", "actual_temperature")
precipitation = read_actual_measurements(raw_folder, "measured_precipitation.csv", "actual_precipitation")
wind_speed = read_actual_measurements(raw_folder, "measured_wind_speed.csv", "actual_wind_speed")

actual = temp.merge(
    precipitation, on="time", validate="one_to_one"
).merge(
    wind_speed, on="time", validate="one_to_one"
)

In [12]:

forecasts = pd.read_csv(raw_folder / "forecast_data_raw.csv")
forecasts["time"] = pd.to_datetime(forecasts["time"])

forecasts = forecasts.pivot(
    index="time",
    columns="model",
    values=weather_variables.keys()
)

forecasts.columns = [
    f"{weather_variables[var]}_{model}"
    for var, model in forecasts.columns
]

forecasts = forecasts.reset_index()

In [24]:

combined = actual.merge(
    forecasts,
    on="time",
    how="inner",
    validate="one_to_one"
)

combined = combined.sort_values("time").reset_index(drop=True)

combined.to_csv(
    processed_folder / "weather_dataset_2024_2026_2.csv",
    index=False
)

print("Final dataset:", combined.shape)
print("Missing values:")
print(combined.isna().sum())

display(combined.head())


Final dataset: (6624, 13)
Missing values:
time                     0
actual_temperature       0
actual_precipitation     0
actual_wind_speed        0
temperature_ECMWF        0
temperature_GFS          0
temperature_ICON_EU      0
precipitation_ECMWF      0
precipitation_GFS        0
precipitation_ICON_EU    0
wind_speed_ECMWF         0
wind_speed_GFS           0
wind_speed_ICON_EU       0
dtype: int64


,time,actual_temperature,actual_precipitation,actual_wind_speed,temperature_ECMWF,temperature_GFS,temperature_ICON_EU,precipitation_ECMWF,precipitation_GFS,precipitation_ICON_EU,wind_speed_ECMWF,wind_speed_GFS,wind_speed_ICON_EU
0,2024-06-01 00:00:00,17.4,0.0,0.8,17.2,15.5,17.4,0.0,1.0,0.0,1.90,2.66,1.04
1,2024-06-01 01:00:00,17.1,0.0,1.2,16.4,15.6,16.5,0.0,0.0,0.0,1.30,2.19,1.13
2,2024-06-01 02:00:00,15.3,0.0,1.3,16.0,15.5,16.0,0.0,0.0,0.0,0.90,1.71,1.08
3,2024-06-01 03:00:00,14.5,0.0,0.7,16.2,15.1,15.6,0.3,0.0,0.0,1.03,2.18,1.00
4,2024-06-01 04:00:00,14.5,0.0,0.9,16.8,15.0,15.3,0.3,0.0,0.0,1.63,2.48,0.61


In [23]:

# How far are the predictions from Tõravere weather station

def km(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 6371 * 2 * math.asin(math.sqrt(a))

for m in weather_models.values():
    base = dict(latitude=lat, longitude=lon, models=m, past_days=1,
                hourly="temperature_2m_previous_day1", timezone="UTC")
    r1 = requests.get(url, params=base).json()
    r2 = requests.get(url, params={**base, "elevation": "nan"}).json()
    t1 = r1["hourly"]["temperature_2m_previous_day1"]
    t2 = r2["hourly"]["temperature_2m_previous_day1"]
    diffs = [a - b for a, b in zip(t1, t2) if a is not None and b is not None]
    print(m,
          f"cell=({r1['latitude']},{r1['longitude']})",
          f"offset={km(lat, lon, r1['latitude'], r1['longitude']):.1f} km",
          f"elev={r1['elevation']} / cell elev(nan)={r2['elevation']}")

ecmwf_ifs025 cell=(58.25,26.5) offset=2.8 km elev=71.0 / cell elev(nan)=75.0
ncep_gfs_global cell=(58.281815,26.484375) offset=2.4 km elev=71.0 / cell elev(nan)=67.0
icon_eu cell=(58.25,26.4375) offset=2.1 km elev=71.0 / cell elev(nan)=67.0
